# CNN for Diabetic Retinopathy Detection — Messidor-2
## MALEEA3 | University of Johannesburg
### Lecturer: Prof. Khmaies Ouahada | Tutor: Mr. Bilal Ahmad

---

**Student:** [Your Name] | **Student No:** [Your Student Number]  
**Task:** Binary classification of diabetic retinopathy from retinal fundus images  
**Classes:**
- **0** — No DR (original grade 0 only)
- **1** — DR Present (original grades 1, 2, 3, 4)

> **Note:** Binary classification was adopted following lecturer guidance after initial
> 4-class grading yielded accuracy below 80%. Binary framing is clinically meaningful
> as a screening tool — it answers the primary triage question:
> does this patient need an ophthalmological referral?

---
## Phase 1 & 2: Dataset Loading and Preprocessing

### Step 1: Install and Import Libraries

In [ ]:
# =============================================================
# BLOCK 1: LIBRARY IMPORTS
# All third-party libraries required for this project are
# imported here at the top so dependencies are clear upfront.
# Uncomment the pip line below and run once if any library
# is missing from your environment.
# =============================================================

# !pip install tensorflow pillow scikit-learn matplotlib seaborn pandas numpy

import os                        # File path operations
import numpy as np               # Numerical array operations
import pandas as pd              # CSV loading and dataframe manipulation
import matplotlib.pyplot as plt  # Plotting training curves, distributions
import seaborn as sns            # Statistical visualisation (heatmaps etc.)

from PIL import Image            # Opening and resizing image files from disk

# scikit-learn: data splitting, class balancing, evaluation metrics
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report,   # Per-class precision, recall, F1
    confusion_matrix,        # True/false positive/negative counts
    ConfusionMatrixDisplay,  # Visualisation wrapper for confusion matrix
    accuracy_score,          # Overall fraction of correct predictions
    roc_auc_score,           # Area under the ROC curve (binary screening metric)
    roc_curve                # TPR vs FPR at all classification thresholds
)

# TensorFlow / Keras: deep learning framework used to build and train the CNN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical  # One-hot label encoding

print('TensorFlow version:', tf.__version__)
print('All libraries loaded successfully.')


### Step 2: Configure Paths and Hyperparameters

In [ ]:
# =============================================================
# BLOCK 2: CONFIGURATION
# All paths and hyperparameters are defined here in one place
# so they are easy to locate and modify without editing deep
# inside the code.
#
# ACTION REQUIRED: Update IMAGE_DIR and CSV_PATH to match
# your local Messidor-2 folder structure before running.
# =============================================================

# Path to the folder containing all .png retinal images
# (this is the 'preprocess' folder inside the Messidor-2 ZIP)
IMAGE_DIR = r'C:\path\to\Messidor-2\preprocess'       # <-- UPDATE THIS

# Path to the CSV file containing image filenames and diagnosis labels
CSV_PATH  = r'C:\path\to\Messidor-2\messidor_data.csv' # <-- UPDATE THIS

# Resize all images to this resolution before feeding into the CNN
# 128x128 balances training speed against spatial detail retention
IMG_SIZE    = (128, 128)

# Fraction of the full dataset held out as the final test set
TEST_SIZE   = 0.20   # 20%

# Fraction of the remaining training data used for validation
VAL_SIZE    = 0.15   # 15% of 80% = ~12% of total

# Random seed ensures reproducible splits and weight initialisation
SEED        = 42

# Number of output classes for binary classification
NUM_CLASSES = 2

# Human-readable class names used in plots and reports
CLASS_NAMES = {0: 'No DR', 1: 'DR Present'}

print('Configuration:')
print(f'  Image size  : {IMG_SIZE}')
print(f'  Test split  : {TEST_SIZE*100:.0f}%')
print(f'  Num classes : {NUM_CLASSES} (Binary)')


### Step 3: Load CSV and Map to Binary Labels

In [ ]:
# =============================================================
# BLOCK 3: LOAD CSV AND INSPECT LABEL DISTRIBUTION
# The Messidor-2 CSV contains two columns:
#   'id_code'   — the image filename (e.g. 'IDRiD_001.png')
#   'diagnosis' — DR severity grade from 0 to 4
# =============================================================

df = pd.read_csv(CSV_PATH)

print('CSV shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())

print('\nOriginal diagnosis value counts (5 grades):')
print(df['diagnosis'].value_counts().sort_index())


In [ ]:
# =============================================================
# BLOCK 4: BINARY LABEL MAPPING
# The original 5 grades (0-4) are remapped to 2 classes:
#
#   Grade 0          --> Class 0: No DR
#   Grades 1, 2, 3, 4 --> Class 1: DR Present (any severity)
#
# Clinical rationale: the primary screening decision is binary —
# does this patient require ophthalmological referral (yes/no)?
# All non-zero grades indicate some level of DR requiring referral,
# regardless of severity grade.
#
# Academic rationale: binary classification was adopted following
# lecturer guidance after 4-class grading yielded accuracy < 80%.
# =============================================================

# Apply the binary mapping using a lambda function row by row
# lambda x: 0 if x == 0 else 1  means:
#   if diagnosis is 0, return 0 (No DR), otherwise return 1 (DR Present)
df['label'] = df['diagnosis'].apply(lambda x: 0 if x == 0 else 1)

print('Binary label distribution after mapping:')
for cls_id, cls_name in CLASS_NAMES.items():
    count = (df['label'] == cls_id).sum()
    print(f'  Class {cls_id} — {cls_name}: {count} images ({count/len(df)*100:.1f}%)')

# =============================================================
# VISUALISE: Plot original 5-grade and binary distributions
# side by side to show the effect of the label mapping
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left plot: original 5-grade distribution
orig = df['diagnosis'].value_counts().sort_index()
axes[0].bar(range(len(orig)), orig.values, color='steelblue', edgecolor='black', alpha=0.85)
axes[0].set_xticks(range(5))
axes[0].set_xticklabels(['0-No DR','1-Mild','2-Mod.','3-Severe','4-Prolif.'], rotation=15)
axes[0].set_title('Original Distribution (5 grades)', fontweight='bold')
axes[0].set_ylabel('Number of Images')
for i, v in enumerate(orig.values):
    axes[0].text(i, v+2, str(v), ha='center', fontsize=9)

# Right plot: binary distribution after mapping
binary = df['label'].value_counts().sort_index()
bars = axes[1].bar([0, 1], binary.values, color=['#2ecc71','#e74c3c'], edgecolor='black', alpha=0.85)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['0 — No DR', '1 — DR Present'], fontsize=10)
axes[1].set_title('Binary Label Distribution', fontweight='bold')
axes[1].set_ylabel('Number of Images')
for bar, v in zip(bars, binary.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+2, str(v), ha='center', fontsize=9)

plt.suptitle('Messidor-2: Label Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')


### Step 4: Load and Preprocess Images

In [ ]:
# =============================================================
# BLOCK 5: IMAGE LOADING AND PREPROCESSING
# Each image is loaded from disk and processed through three steps:
#   1. Convert to RGB — ensures every image has exactly 3 channels
#      (some PNG files may be saved as RGBA with 4 channels)
#   2. Resize to 128x128 — all images must be the same dimensions
#      for the CNN to process them in batches
#   3. Normalise pixels — dividing by 255 maps integer pixel values
#      from [0, 255] to floating-point [0.0, 1.0]. This is essential
#      for stable gradient flow during backpropagation, as the CNN's
#      initial weights are small (~0.01) and large inputs would cause
#      unstable gradient magnitudes.
# =============================================================

images  = []   # Accumulates processed image arrays
labels  = []   # Accumulates corresponding binary labels
skipped = []   # Tracks any image files not found on disk

print(f'Loading images from: {IMAGE_DIR}')
print(f'Resizing to: {IMG_SIZE[0]} x {IMG_SIZE[1]} pixels')
print('This may take a minute...\n')

for idx, row in df.iterrows():
    # Build the full file path from the folder and CSV filename
    img_path = os.path.join(IMAGE_DIR, row['id_code'])

    # Skip images whose files are not found on disk
    if not os.path.exists(img_path):
        skipped.append(row['id_code'])
        continue

    # Open image, force 3-channel RGB, resize to target dimensions
    img = Image.open(img_path).convert('RGB')
    img = img.resize(IMG_SIZE)  # PIL resize takes (width, height)

    # Convert PIL image to a NumPy float array and normalise to [0, 1]
    img_array = np.array(img, dtype=np.float32) / 255.0

    images.append(img_array)
    labels.append(row['label'])

# Convert Python lists to NumPy arrays for efficient batch processing
X = np.array(images, dtype=np.float32)  # Shape: (N, 128, 128, 3)
y = np.array(labels, dtype=np.int32)    # Shape: (N,)

print(f'Successfully loaded : {len(images)} images')
print(f'Skipped (not found) : {len(skipped)}')
print(f'X shape : {X.shape}  — (samples, height, width, channels)')
print(f'y shape : {y.shape}')
print(f'Pixel value range   : [{X.min():.3f}, {X.max():.3f}]')


In [ ]:
# =============================================================
# BLOCK 6: VISUALISE SAMPLE IMAGES
# Display 5 examples from each class (No DR and DR Present)
# to visually confirm images were loaded and normalised correctly
# and to illustrate the visual difference between the two classes.
# =============================================================

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Retinal Fundus Images — Messidor-2 (Binary)',
             fontsize=13, fontweight='bold')

# Find indices of the first 5 images from each class
no_dr_idx = np.where(y == 0)[0][:5]  # First 5 No DR images
dr_idx    = np.where(y == 1)[0][:5]  # First 5 DR Present images

# Row 0: No DR examples
for i, idx in enumerate(no_dr_idx):
    axes[0, i].imshow(X[idx])
    axes[0, i].set_title('No DR', fontsize=9, color='green', fontweight='bold')
    axes[0, i].axis('off')

# Row 1: DR Present examples
for i, idx in enumerate(dr_idx):
    axes[1, i].imshow(X[idx])
    axes[1, i].set_title('DR Present', fontsize=9, color='red', fontweight='bold')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sample_images.png')


### Step 5: Train / Validation / Test Split

In [ ]:
# =============================================================
# BLOCK 7: STRATIFIED DATASET SPLITTING
# The dataset is divided into three non-overlapping subsets:
#
#   Training set (~68%)   — Images the model learns from.
#                           Weights are updated based on these.
#   Validation set (~12%) — Monitors generalisation after each
#                           epoch. Not used for weight updates.
#                           Used by EarlyStopping and ReduceLROnPlateau.
#   Test set (20%)        — Completely held out until final evaluation.
#                           Provides an unbiased performance estimate.
#
# stratify=y ensures each split preserves the same class proportion
# as the full dataset. Without stratification, a split could
# accidentally contain very few DR Present images, making the
# evaluation unreliable.
# =============================================================

# Split 1: Reserve 20% as the final test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# Split 2: From the remaining 80%, use 15% for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=VAL_SIZE, random_state=SEED, stratify=y_train_val
)

print('Dataset splits (stratified):')
print(f'  Training   : {X_train.shape[0]} images ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'  Validation : {X_val.shape[0]}  images ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'  Test       : {X_test.shape[0]}  images ({X_test.shape[0]/len(X)*100:.1f}%)')

# Confirm class proportions are preserved in each split
for split_name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    no_dr = (y_split==0).sum()
    dr    = (y_split==1).sum()
    print(f'  {split_name:5s}: No DR={no_dr}  DR Present={dr}')


In [ ]:
# =============================================================
# BLOCK 8: CLASS WEIGHT COMPUTATION
# Messidor-2 is imbalanced — No DR images are the majority.
# Without correction, the model learns to predict No DR for
# almost everything, achieving deceptively high accuracy while
# failing to detect actual DR cases (clinically useless).
#
# Class weights correct this by scaling the loss contribution
# of each sample: rarer classes receive higher weights so their
# misclassification is penalised more heavily during training.
#
# Formula: weight(class i) = total_samples / (n_classes * count(class i))
# =============================================================

cw_array = class_weight.compute_class_weight(
    class_weight='balanced',   # Automatically computes inverse-frequency weights
    classes=np.array([0, 1]),  # The two binary classes
    y=y_train                  # Computed from training labels only
)
# Convert array to dictionary format required by model.fit()
class_weight_dict = {0: cw_array[0], 1: cw_array[1]}

print('Class weights (higher = penalised more for misclassification):')
for cls_id, w in class_weight_dict.items():
    print(f'  Class {cls_id} ({CLASS_NAMES[cls_id]}): {w:.4f}')


In [ ]:
# =============================================================
# BLOCK 9: ONE-HOT ENCODING
# The categorical_crossentropy loss function requires labels in
# one-hot vector format rather than plain integers.
#
# One-hot encoding converts each integer class label into a
# binary vector of length NUM_CLASSES:
#   Class 0 (No DR)      --> [1, 0]
#   Class 1 (DR Present) --> [0, 1]
#
# The loss function then compares these vectors against the
# model's softmax output [P(No DR), P(DR Present)].
# =============================================================

y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   num_classes=NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  num_classes=NUM_CLASSES)

print(f'y_train_cat shape: {y_train_cat.shape}  — (samples, 2)')
print(f'Example encoding: label {y_train[0]} --> {y_train_cat[0]}')


In [ ]:
# =============================================================
# BLOCK 10: SAVE PREPROCESSED ARRAYS TO DISK
# Saving the preprocessed arrays means Phase 3 can be run
# independently without reloading and reprocessing all images.
# numpy .npy format is efficient and preserves array dtype exactly.
# =============================================================

np.save('X_train.npy',     X_train)     # Training images
np.save('X_val.npy',       X_val)       # Validation images
np.save('X_test.npy',      X_test)      # Test images
np.save('y_train.npy',     y_train)     # Training labels (integers)
np.save('y_val.npy',       y_val)       # Validation labels (integers)
np.save('y_test.npy',      y_test)      # Test labels (integers)
np.save('y_train_cat.npy', y_train_cat) # Training labels (one-hot)
np.save('y_val_cat.npy',   y_val_cat)   # Validation labels (one-hot)
np.save('y_test_cat.npy',  y_test_cat)  # Test labels (one-hot)

print('All preprocessed arrays saved to disk.')
print('\n✓ Phase 1 & 2 complete. Proceed to Phase 3.')


---
## Phase 3: CNN Architecture, Training, and Evaluation

**Architecture note:** The binary model is identical to the initial 4-class model
except the output Dense layer uses 2 units instead of 4.
All 10 layers, filter counts, and hyperparameters are unchanged.


### Step 6: Reload Preprocessed Data (skip if running continuously)

In [ ]:
# =============================================================
# BLOCK 11: RELOAD PREPROCESSED DATA
# Run this cell ONLY if starting Phase 3 in a fresh kernel
# without having run Phase 1 & 2 first.
# If you are running the notebook top-to-bottom, skip this cell
# as all arrays are already in memory from the previous steps.
# =============================================================

X_train     = np.load('X_train.npy')
X_val       = np.load('X_val.npy')
X_test      = np.load('X_test.npy')
y_train     = np.load('y_train.npy')
y_val       = np.load('y_val.npy')
y_test      = np.load('y_test.npy')
y_train_cat = np.load('y_train_cat.npy')
y_val_cat   = np.load('y_val_cat.npy')
y_test_cat  = np.load('y_test_cat.npy')

# Restore class configuration constants
NUM_CLASSES = 2
CLASS_NAMES = {0: 'No DR', 1: 'DR Present'}

# Recompute class weights from the reloaded training labels
cw_array = class_weight.compute_class_weight(
    'balanced', classes=np.array([0,1]), y=y_train)
class_weight_dict = {0: cw_array[0], 1: cw_array[1]}

print(f'Data reloaded:')
print(f'  X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')


### Step 7: Build the 10-Layer CNN

In [ ]:
# =============================================================
# BLOCK 12: CNN ARCHITECTURE — 10-LAYER BINARY CLASSIFIER
#
# The model consists of four convolutional blocks followed by
# a fully connected classification head:
#
#   FEATURE EXTRACTION BACKBONE (Layers 1-8):
#   Four blocks, each containing:
#     - Conv2D: learns spatial filters to detect retinal patterns
#     - MaxPooling2D: reduces spatial dimensions, retains strongest signals
#   Filter counts double per block (32->64->128->256) to capture
#   increasingly complex features in deeper layers.
#
#   CLASSIFICATION HEAD (Layers 9-10 + Output):
#     - Flatten (Layer 9): reshapes 8x8x256 feature maps to a 16,384
#       element vector for input to Dense layers
#     - Dropout (regularisation): randomly zeros 50% of neurons
#       during training to prevent overfitting
#     - Dense 256 (Layer 10): fully connected layer that combines
#       all extracted features into a task-specific representation
#     - Dense 2, Softmax (Output): produces [P(No DR), P(DR Present)]
#       probability distribution. Predicted class = argmax of output.
#
# COMPILATION:
#   - Adam optimiser (lr=1e-3): adaptive learning rate, robust default
#   - categorical_crossentropy: standard loss for one-hot multi-class labels
#   - accuracy: primary monitoring metric during training
# =============================================================

# Derive input shape from training data (128, 128, 3)
input_shape = X_train.shape[1:]

model = models.Sequential(name='DR_CNN_Binary', layers=[

    # --- Block 1: Low-level feature extraction (edges, colour gradients) ---
    layers.Conv2D(32, (3,3), activation='relu', padding='same',
                  input_shape=input_shape, name='Conv1'),
    layers.MaxPooling2D((2,2), name='Pool1'),  # 128x128 -> 64x64

    # --- Block 2: Mid-level feature extraction (textures, simple shapes) ---
    layers.Conv2D(64, (3,3), activation='relu', padding='same', name='Conv2'),
    layers.MaxPooling2D((2,2), name='Pool2'),  # 64x64 -> 32x32

    # --- Block 3: Complex feature extraction (lesion-like structures) ---
    layers.Conv2D(128, (3,3), activation='relu', padding='same', name='Conv3'),
    layers.MaxPooling2D((2,2), name='Pool3'),  # 32x32 -> 16x16

    # --- Block 4: High-level retinal feature extraction ---
    layers.Conv2D(256, (3,3), activation='relu', padding='same', name='Conv4'),
    layers.MaxPooling2D((2,2), name='Pool4'),  # 16x16 -> 8x8

    # --- Classification head ---
    layers.Flatten(name='Flatten'),        # 8x8x256 = 16,384 values -> 1D vector
    layers.Dropout(0.5, name='Dropout'),   # Regularisation: zeros 50% of neurons
    layers.Dense(256, activation='relu', name='FC1'),  # Fully connected layer

    # Output: 2 softmax units produce binary class probabilities
    layers.Dense(NUM_CLASSES, activation='softmax', name='Output')
])

# Compile the model with optimiser, loss function, and evaluation metric
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',  # Appropriate for one-hot labels + softmax
    metrics=['accuracy']
)

# Print architecture summary showing layer names, output shapes, and parameters
model.summary()


### Step 8: Train the Model

In [ ]:
# =============================================================
# BLOCK 13: MODEL TRAINING WITH CALLBACKS
#
# Two callbacks govern the training schedule:
#
#   EarlyStopping:
#     Monitors validation loss after each epoch. If it does not
#     improve for 10 consecutive epochs (patience=10), training
#     halts and the best weights are restored. This prevents the
#     model from overfitting past its optimal generalisation point.
#
#   ReduceLROnPlateau:
#     If validation loss stagnates for 5 epochs (patience=5),
#     the learning rate is halved (factor=0.5). This allows the
#     model to make finer weight adjustments near the optimum.
#     The minimum learning rate is capped at 1e-6.
#
# class_weight=class_weight_dict scales the loss contribution
# of each sample, penalising DR Present misclassification more
# heavily to counter the No DR majority class bias.
# =============================================================

# EarlyStopping: halt training when val_loss stops improving
early_stop = EarlyStopping(
    monitor='val_loss',          # Watch validation loss
    patience=10,                 # Stop after 10 epochs of no improvement
    restore_best_weights=True,   # Revert to the epoch with lowest val_loss
    verbose=1                    # Print a message when triggered
)

# ReduceLROnPlateau: reduce learning rate when training stagnates
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',  # Watch validation loss
    factor=0.5,          # Multiply learning rate by 0.5 when triggered
    patience=5,          # Trigger after 5 epochs of no improvement
    min_lr=1e-6,         # Never reduce learning rate below this value
    verbose=1
)

EPOCHS     = 50   # Maximum number of training epochs
BATCH_SIZE = 32   # Number of images processed per gradient update

print(f'Training binary CNN — up to {EPOCHS} epochs, batch size {BATCH_SIZE}')
print('EarlyStopping will halt training if val_loss does not improve for 10 epochs.\n')

# Train the model
# validation_data is evaluated after each epoch but never used for weight updates
history = model.fit(
    X_train, y_train_cat,                    # Training images and one-hot labels
    validation_data=(X_val, y_val_cat),      # Validation set for monitoring
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,          # Apply class balancing weights
    callbacks=[early_stop, reduce_lr],       # Attach both callbacks
    verbose=1                                # Print per-epoch progress
)

print('\nTraining complete.')


### Step 9: Plot Training and Validation Curves

In [ ]:
# =============================================================
# BLOCK 14: TRAINING AND VALIDATION CURVES
# Plots accuracy and loss for both training and validation sets
# over all epochs. These curves reveal:
#   - Whether the model is learning (both curves improving)
#   - Whether it is overfitting (train improves, val degrades)
#   - Whether it is underfitting (both curves plateau early)
# Solid lines = training metrics; dashed lines = validation metrics.
# =============================================================

epochs_ran = len(history.history['loss'])  # Actual epochs before early stopping

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left plot: accuracy over epochs
axes[0].plot(history.history['accuracy'],
             label='Train Accuracy', color='steelblue', linewidth=2)
axes[0].plot(history.history['val_accuracy'],
             label='Val Accuracy', color='darkorange', linewidth=2, linestyle='--')
axes[0].set_title('Model Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])  # Fix y-axis to [0,1] for consistent comparison

# Right plot: loss over epochs
axes[1].plot(history.history['loss'],
             label='Train Loss', color='steelblue', linewidth=2)
axes[1].plot(history.history['val_loss'],
             label='Val Loss', color='darkorange', linewidth=2, linestyle='--')
axes[1].set_title('Model Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Categorical Cross-Entropy Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Binary CNN Training History ({epochs_ran} epochs ran)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')


### Step 10: Evaluate on the Test Set

In [ ]:
# =============================================================
# BLOCK 15: TEST SET EVALUATION
# The model is evaluated on the held-out test set — images it
# has never seen during training or validation.
#
# model.evaluate() returns the loss and accuracy.
# model.predict() returns softmax probabilities for each class.
# np.argmax(..., axis=1) converts probabilities to predicted
# class indices (0 or 1) by selecting the highest probability.
# =============================================================

# Evaluate loss and accuracy on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)

# Generate predictions: softmax output shape = (n_test, 2)
y_pred_probs = model.predict(X_test, verbose=0)

# Convert probabilities to class labels: take the index of the highest probability
y_pred = np.argmax(y_pred_probs, axis=1)

# Ground truth integer labels for comparison
y_true = y_test

print('=' * 50)
print('TEST SET RESULTS')
print('=' * 50)
print(f'  Test Accuracy : {test_accuracy*100:.2f}%')
print(f'  Test Loss     : {test_loss:.4f}')
print('=' * 50)


In [ ]:
# =============================================================
# BLOCK 16: CLASSIFICATION REPORT
# Generates a detailed per-class performance breakdown:
#   Precision: of all images predicted as this class, what fraction
#              were actually this class? (TP / (TP + FP))
#   Recall:    of all images truly this class, what fraction did
#              the model correctly identify? (TP / (TP + FN))
#              Recall for DR Present is the most critical clinical metric.
#   F1-Score:  harmonic mean of precision and recall.
#              Best single metric for imbalanced classification.
# =============================================================

print('CLASSIFICATION REPORT')
print('=' * 55)
print(classification_report(
    y_true, y_pred,
    target_names=['No DR', 'DR Present']
))


In [ ]:
# =============================================================
# BLOCK 17: CONFUSION MATRIX
# The confusion matrix shows exactly where the model is making
# errors. For binary classification the matrix is 2x2:
#
#                    Predicted: No DR  Predicted: DR Present
#   True: No DR      [ TN             FP (false alarm)      ]
#   True: DR Present [ FN (missed!)   TP (correct detect.)  ]
#
# The normalised version divides each row by the true class count,
# showing the proportion correctly/incorrectly classified per class.
# =============================================================

cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: raw counts confusion matrix
ConfusionMatrixDisplay(cm, display_labels=['No DR','DR Present']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

# Right: row-normalised confusion matrix (proportions within each true class)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=['No DR','DR Present']).plot(
    ax=axes[1], colorbar=False, cmap='Blues', values_format='.2f')
axes[1].set_title('Confusion Matrix (Normalised)', fontweight='bold')

plt.suptitle('Binary CNN — Confusion Matrices on Test Set',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')


In [ ]:
# =============================================================
# BLOCK 18: ROC CURVE AND AUC SCORE
# The Receiver Operating Characteristic (ROC) curve is the
# standard evaluation metric for binary medical screening models.
#
# It plots True Positive Rate (Recall) vs False Positive Rate
# at every possible classification threshold (not just 0.5).
#
# AUC (Area Under the Curve) summarises the curve in one number:
#   AUC = 1.0 -> perfect classifier
#   AUC = 0.5 -> random guessing (diagonal line)
#   AUC = 0.7605 (this model) -> meaningful discrimination
#
# y_pred_probs[:, 1] extracts the probability of DR Present
# for each test image, which is used to compute the ROC curve.
# =============================================================

# Extract DR Present probability column from softmax output
dr_probs = y_pred_probs[:, 1]

# Compute AUC score and ROC curve points
auc_score = roc_auc_score(y_true, dr_probs)
fpr, tpr, _ = roc_curve(y_true, dr_probs)  # fpr=x-axis, tpr=y-axis

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2.5,
         label=f'CNN Model (AUC = {auc_score:.4f})')
plt.plot([0,1],[0,1], color='gray', linewidth=1, linestyle='--',
         label='Random Classifier (AUC = 0.50)')
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12)
plt.title('ROC Curve — Binary DR Screening', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'ROC-AUC Score: {auc_score:.4f}')
print('Saved: roc_curve.png')


### Step 11: Sample Test Predictions

In [ ]:
# =============================================================
# BLOCK 19: SAMPLE TEST PREDICTIONS
# Displays 16 randomly selected test images with their true
# label (from the CSV) and the model's predicted label with
# confidence percentage.
#
# Border colour coding:
#   Green = correct prediction (true label matches predicted label)
#   Red   = incorrect prediction (model made an error)
#
# This provides a qualitative view of model performance and shows
# what kinds of images the model struggles with.
# =============================================================

n_show  = 16
# Randomly select 16 indices from the test set without replacement
indices = np.random.choice(len(X_test), n_show, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle(
    'Sample Test Predictions — Binary DR Classification\n'
    'Green border = Correct  |  Red border = Incorrect',
    fontsize=13, fontweight='bold'
)

for i, idx in enumerate(indices):
    ax = axes[i//4, i%4]      # Map flat index to 4x4 grid position
    ax.imshow(X_test[idx])    # Display normalised image (matplotlib handles [0,1])
    ax.axis('off')

    true_cls   = y_true[idx]                          # Ground truth class
    pred_cls   = y_pred[idx]                          # Predicted class
    confidence = y_pred_probs[idx][pred_cls] * 100   # Confidence as percentage
    correct    = (true_cls == pred_cls)               # Boolean: was it right?
    colour     = 'green' if correct else 'red'        # Border colour

    ax.set_title(
        f'True : {CLASS_NAMES[true_cls]}\n'
        f'Pred : {CLASS_NAMES[pred_cls]} ({confidence:.0f}%)',
        fontsize=8, color=colour, fontweight='bold'
    )

    # Apply coloured border to each subplot panel
    for spine in ax.spines.values():
        spine.set_edgecolor(colour)
        spine.set_linewidth(3)
        spine.set_visible(True)

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sample_predictions.png')


### Step 12: Final Summary and Save Model

In [ ]:
# =============================================================
# BLOCK 20: FINAL PERFORMANCE SUMMARY AND MODEL SAVE
# Prints a clean formatted table of all evaluation metrics
# for easy transcription into the research report.
#
# The trained model is saved in Keras native format (.keras)
# which can be reloaded later with:
#   model = keras.models.load_model('messidor2_DR_CNN_binary.keras')
# =============================================================

from sklearn.metrics import precision_score, recall_score, f1_score

print('=' * 60)
print('FINAL MODEL PERFORMANCE SUMMARY — BINARY CLASSIFICATION')
print('=' * 60)
print(f'  Test Accuracy  : {accuracy_score(y_true, y_pred)*100:.2f}%')
print(f'  ROC-AUC Score  : {auc_score:.4f}')
print()
print(f"  {'Class':<18} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print('  ' + '-'*50)

# Compute per-class metrics (average=None returns one value per class)
p = precision_score(y_true, y_pred, average=None, zero_division=0)
r = recall_score(y_true, y_pred, average=None, zero_division=0)
f = f1_score(y_true, y_pred, average=None, zero_division=0)

for cls in range(NUM_CLASSES):
    print(f"  {CLASS_NAMES[cls]:<18} {p[cls]:>10.4f} {r[cls]:>10.4f} {f[cls]:>10.4f}")

print('  ' + '-'*50)
# Macro average: equal weight per class regardless of sample count
print(f"  {'Macro Avg':<18} "
      f"{precision_score(y_true,y_pred,average='macro',zero_division=0):>10.4f} "
      f"{recall_score(y_true,y_pred,average='macro',zero_division=0):>10.4f} "
      f"{f1_score(y_true,y_pred,average='macro',zero_division=0):>10.4f}")
print('=' * 60)

# Save trained model weights and architecture to disk
model.save('messidor2_DR_CNN_binary.keras')
print('\nModel saved: messidor2_DR_CNN_binary.keras')
print('\n✓ All phases complete! Output files generated:')
for f_name in ['class_distribution.png', 'sample_images.png',
               'training_curves.png', 'confusion_matrix.png',
               'roc_curve.png', 'sample_predictions.png',
               'messidor2_DR_CNN_binary.keras']:
    print(f'  {f_name}')
